<a href="https://www.kaggle.com/code/dnkumars/equipment-monitoring-ann?scriptVersionId=350014954" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/dnkumars/equipment-monitoring-ann?scriptVersionId=212300493" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<p style="font-family: 'Comic Sans MS', serif; font-size: 40px; font-weight: bold; text-align: center; color: #D4EBF8; background-color: #212529; padding: 20px; border: 2px solid #D4EBF8; border-radius: 15px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);">Equipment Monitoring🖲️|ANN 🧠</p>


<div align="center" style="background-color: #091057; padding: 20px; border-radius: 10px;">
  <h1 style="color: #A0D683;">Loading Libraries</h1>
</div>

In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from sklearn.ensemble import IsolationForest
from IPython.display import IFrame
from plotly.offline import plot

<div align="center" style="background-color: #77CDFF; padding: 20px; border-radius: 10px;">
  <h1 style="color: D9EAFD5;">Data Loading</h1>
</div>

In [2]:
data = pd.read_csv("/kaggle/input/industrial-equipment-monitoring-dataset/equipment_anomaly_data.csv")

In [3]:
data.head()

,temperature,pressure,vibration,humidity,equipment,location,faulty
0,58.180180,25.029278,0.606516,45.694907,Turbine,Atlanta,0.0
1,75.740712,22.954018,2.338095,41.867407,Compressor,Chicago,0.0
2,71.358594,27.276830,1.389198,58.954409,Turbine,San Francisco,0.0
3,71.616985,32.242921,1.770690,40.565138,Pump,Atlanta,0.0
4,66.506832,45.197471,0.345398,43.253795,Pump,New York,0.0


In [4]:
data.isnull().sum()

temperature    0
pressure       0
vibration      0
humidity       0
equipment      0
location       0
faulty         0
dtype: int64

<div align="center" style="background-color: #72BF78; padding: 20px; border-radius: 10px;">
  <h1 style="color: #433878;">Exploratory Data Analysis</h1>

</div>

## Map

In [5]:
city_coordinates = {
    "New York": [40.7128, -74.0060],
    "Houston": [29.7604, -95.3698],
    "Chicago": [41.8781, -87.6298],
    "San Francisco": [37.7749, -122.4194],
    "Atlanta": [33.7490, -84.3880]
}
data["city"] = data["location"] 
data["latitude"] = data["city"].map(lambda x: city_coordinates[x][0])
data["longitude"] = data["city"].map(lambda x: city_coordinates[x][1])
m = folium.Map(location=[37.0902, -95.7129], zoom_start=4)
marker_cluster = MarkerCluster().add_to(m)
for i, row in data.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=f"City: {row['city']}, Equipment: {row['equipment']}"
    ).add_to(marker_cluster)

m

## Univariate Analysis

In [6]:
fig1 = px.histogram(data, x="temperature", title="Temperature Distribution")
filename="hist1.html"
plot(fig1, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=600))

In [7]:
fig2 = px.box(data, y='humidity',color='equipment', title="Humidity Distribution")
filename="box.html"
plot(fig2, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=600))

## Bivariate Analysis

In [8]:
fig3 = px.scatter(data, x="temperature", y="pressure", color="faulty", title="Temperature vs Pressure")
filename="scatter1.html"
plot(fig3, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=600))

In [9]:
data.columns

Index(['temperature', 'pressure', 'vibration', 'humidity', 'equipment',
       'location', 'faulty', 'city', 'latitude', 'longitude'],
      dtype='object')

In [10]:
corr_matrix = data.drop(['city', 'latitude', 'longitude', 'equipment','location'],axis=1).corr()
fig4 = px.imshow(corr_matrix, text_auto=True, title="Correlation Heatmap")
filename="corr.html"
plot(fig4, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=800))

<div align="center" style="background-color: #FFE3E3; padding: 20px; border-radius: 10px;">
  <h1 style="color: #091057;">Data Preprocessing</h1>

</div>

In [11]:
label_encoder = LabelEncoder()
data['equipment'] = label_encoder.fit_transform(data['equipment'])
data['location'] = label_encoder.fit_transform(data['location'])

In [12]:
data.head()

,temperature,pressure,vibration,humidity,equipment,location,faulty,city,latitude,longitude
0,58.180180,25.029278,0.606516,45.694907,2,0,0.0,Atlanta,33.7490,-84.3880
1,75.740712,22.954018,2.338095,41.867407,0,1,0.0,Chicago,41.8781,-87.6298
2,71.358594,27.276830,1.389198,58.954409,2,4,0.0,San Francisco,37.7749,-122.4194
3,71.616985,32.242921,1.770690,40.565138,1,0,0.0,Atlanta,33.7490,-84.3880
4,66.506832,45.197471,0.345398,43.253795,1,3,0.0,New York,40.7128,-74.0060


In [13]:
scaler = StandardScaler()
numerical_cols = ['temperature', 'pressure', 'vibration', 'humidity']
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

In [14]:
data.head()

,temperature,pressure,vibration,humidity,equipment,location,faulty,city,latitude,longitude
0,-0.786610,-1.031582,-1.379925,-0.364984,2,0,0.0,Atlanta,33.7490,-84.3880
1,0.297440,-1.231493,0.996943,-0.688233,0,1,0.0,Chicago,41.8781,-87.6298
2,0.026922,-0.815074,-0.305569,0.754840,2,4,0.0,San Francisco,37.7749,-122.4194
3,0.042873,-0.336688,0.218089,-0.798215,1,0,0.0,Atlanta,33.7490,-84.3880
4,-0.272588,0.911232,-1.738352,-0.571147,1,3,0.0,New York,40.7128,-74.0060


<div align="center" style="background-color: #FFD7C4; padding: 20px; border-radius: 10px;">
  <h1 style="color: #001F3F;">Model</h1>

</div>

In [15]:
X = data.drop(columns=["faulty",'city'])
y = data["faulty"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [17]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      1377
         1.0       0.93      0.88      0.90       158

    accuracy                           0.98      1535
   macro avg       0.96      0.94      0.95      1535
weighted avg       0.98      0.98      0.98      1535



In [18]:
cm = confusion_matrix(y_test, y_pred)
fig5 = go.Figure(data=go.Heatmap(
    z=cm,
    x=["Predicted 0", "Predicted 1"],  
    y=["Actual 0", "Actual 1"],
    colorscale='Viridis',
    text=np.round(cm, 2),  
    hoverinfo="z"  
))
fig5.update_traces(
    texttemplate="%{text}",
    textfont={"size": 12},
    showscale=True 
)

fig5.update_layout(
    title="Confusion Matrix Heatmap",
    xaxis_title="Predicted Label",
    yaxis_title="True Label",
    autosize=True)
filename = "cm.html"
plot(fig5, filename=filename, auto_open=False)
IFrame(filename, width=800, height=600)

## Clustering

In [19]:
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X)
feature_x = 'temperature'
feature_y = 'pressure'
fig7 = px.scatter(
    X, 
    x=feature_x, 
    y=feature_y, 
    color=clusters.astype(str),  
    title=f"KMeans Clustering ({feature_x} vs {feature_y})",
    labels={feature_x: feature_x.capitalize(), feature_y: feature_y.capitalize()}
)
filename = "clust_no_pca.html"
plot(fig7, filename=filename, auto_open=False)
IFrame(filename, width=800, height=600)

/opt/conda/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning:

The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning



## Anomaly Detection

In [20]:
iso_forest = IsolationForest(contamination=0.1, random_state=42)
data["anomaly"] = iso_forest.fit_predict(X)
fig8 = px.scatter(data, x="temperature", y="pressure", color="anomaly", title="Anomaly Detection")
filename="ano.html"
plot(fig8, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=600))

/opt/conda/lib/python3.10/site-packages/sklearn/base.py:439: UserWarning:

X does not have valid feature names, but IsolationForest was fitted with feature names



In [21]:
model = Sequential([
    Input(shape=(X_train.shape[1],)), 
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=16, verbose=1)
loss, accuracy = model.evaluate(X_test, y_test)

Epoch 1/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7548 - loss: 2.0176 - val_accuracy: 0.9023 - val_loss: 0.2795
Epoch 2/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9110 - loss: 0.2746 - val_accuracy: 0.9324 - val_loss: 0.2310
Epoch 3/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9259 - loss: 0.2489 - val_accuracy: 0.9357 - val_loss: 0.2127
Epoch 4/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9346 - loss: 0.2351 - val_accuracy: 0.9430 - val_loss: 0.2099
Epoch 5/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9458 - loss: 0.2024 - val_accuracy: 0.9471 - val_loss: 0.2077
Epoch 6/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9422 - loss: 0.2043 - val_accuracy: 0.9381 - val_loss: 0.2028
Epoch 7/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9481 - loss: 0.1902 - val_accuracy: 0.9438 - val_loss: 0.1786
Epoch 8/50
307/307 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9551 - loss: 0.1696 - val_accuracy: 0.

In [22]:
print(f"Test Accuracy: {accuracy}")

Test Accuracy: 0.9745928049087524


In [23]:
fig10 = go.Figure()
fig10.add_trace(go.Scatter(y=history.history['accuracy'], mode='lines', name='Training Accuracy'))
fig10.add_trace(go.Scatter(y=history.history['val_accuracy'], mode='lines', name='Validation Accuracy'))
fig10.update_layout(title="Model Accuracy Over Epochs", xaxis_title="Epochs", yaxis_title="Accuracy")
filename="ann.html"
plot(fig10, filename=filename, auto_open=False)
display(IFrame(filename, width=800, height=600))

<div align="center" style="background-color: #C4E1F6; padding: 20px; border-radius: 10px;">
  <h1 style="color: blue;">Thank You 🙇‍♂️ for Visiting My Notebook!</h1>

  <p style="font-size: 18px; color: black;">
    If you found this content valuable, please consider giving it a upvote <span style="color: blue;">👍</span>.
    <br>Your support is greatly appreciated and motivates me to continue developing more valuable and informative notebooks
  </p>
</div>
